# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Greemines/Flyrank-Notebook-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
print(duckdb.__version__)

1.3.2


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN is not None)

True


In [3]:
import duckdb

con = duckdb.connect()

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ Connected to Hugging Face")

✅ Connected to Hugging Face


In [5]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance_sample.parquet'
)
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     11694072 │
└──────────────┘

In [6]:
#testing the connection
con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance_sample.parquet'
)
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

In [13]:
#checking the existing columns
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
)

for f in files[:30]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [14]:
march = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

march

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [15]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content page for one client on one report date.

This assignment uses the March 2026 partition of the fact_content_daily_performance table to verify the data contract and build features.

My lanes goal (CTR / Engagement Opportunity Scoring) is to rank pages that have the greatest opportunity to improve click-through rate (CTR) and user engagement using historical information available before the decision is made.

In [17]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

┌────────────┬────────────┬────────────┐
│ total_rows │ first_day  │  last_day  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_impressions
- gsc_clicks
- gsc_sum_position
- scroll_events
- sessions_ai

These are historical measurements available before making a CTR or engagement improvement decision.

## Label / Proxy

The model ranks pages according to CTR / engagement opportunity using historical search performance.

## Context

- client_hash_id
- content_hash_id
- report_date

These fields identify or group records but should not be used as learning features.

## Excluded

- Future performance information because it would leak future knowledge.
- Availability flags are used only for filtering.
- Any columns derived from the prediction target are excluded to prevent data leakage.

In [18]:
# This cell is for CODE (numbers, a query, a check).

con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
#to check grain
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬─────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ duplicate_count │
│    date     │    varchar     │     varchar     │      int64      │
├─────────────┴────────────────┴─────────────────┴─────────────────┤
│                              0 rows                              │
└──────────────────────────────────────────────────────────────────┘

In [20]:
#to check availability
con.sql(f"""
SELECT
    COUNT(*) AS usable_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┐
│ usable_rows │
│    int64    │
├─────────────┤
│      364347 │
└─────────────┘

In [21]:
#feature frame
feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    scroll_events,
    sessions_ai
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
LIMIT 10;
""")

feature_frame

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────┬─────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ scroll_events │ sessions_ai │
│    date     │         varchar         │         varchar          │      int64      │   int64    │      int64       │     int64     │    int64    │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼──────────────────┼───────────────┼─────────────┤
│ 2026-03-01  │ client_65de48885f4ef01b │ content_5c80451459c29b4a │               5 │          0 │               27 │             0 │           0 │
│ 2026-03-01  │ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │              39 │          0 │              221 │             0 │           1 │
│ 2026-03-01  │ client_65de48885f4ef01b │ content_e25ea7297a1dffd3 │             179 │          0 │       

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This analysis only uses data from the March 2026 partition of the warehouse.

Different clients have different amounts of historical data, so the available history is not balanced across all clients.

Rows are filtered using gsc_data_available IS TRUE and ga4_data_available IS TRUE, meaning records with missing or unavailable data are excluded.

This dataset can identify pages with CTR and engagement opportunities based on historical performance, but it cannot explain why user behavior occurred or guarantee future performance.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.